In [1]:
import os, json, time
from dotenv import load_dotenv
from groq import Groq
from beir.datasets.data_loader import GenericDataLoader

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.3-70b-versatile"

def has_foreign(s):
    for ch in str(s):
        o = ord(ch)
        if o < 128: continue
        if 0x0600 <= o <= 0x06FF: continue
        if 0x0750 <= o <= 0x077F: continue
        if 0xFB50 <= o <= 0xFDFF: continue
        if 0xFE70 <= o <= 0xFEFF: continue
        if ch in '‘’“”—–…•&\u200c\u200d\u060c\u061f': continue
        return True
    return False

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

print("Foreign-character scan across ALL queries:")
for ds in ["scifact", "nfcorpus"]:
    data = load_jsonl(f"queries/{ds}_cs.jsonl")
    flagged = [r["id"] for r in data if has_foreign(r["ur_en"]) or has_foreign(r["roman"])]
    print(f"  {ds}: {len(flagged)}/{len(data)} flagged -> {flagged}")

f:\Research Paper\AI2ML code-switching benchmark\venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Foreign-character scan across ALL queries:
  scifact: 24/300 flagged -> ['100', '238', '261', '327', '560', '628', '692', '693', '759', '781', '783', '820', '821', '847', '1107', '1130', '1140', '1180', '1185', '1278', '1292', '1303', '1332', '1352']
  nfcorpus: 4/323 flagged -> ['PLAIN-2500', 'PLAIN-2730', 'PLAIN-3261', 'PLAIN-3352']


In [2]:
BATCH_PROMPT = """You are a native Urdu speaker from Pakistan. For EACH numbered English query, rewrite it in TWO ways a real Pakistani would type it.

CRITICAL RULES:
- DO NOT add, expand, or invent meaning. If the query is a short keyword, title, or phrase, keep it EQUALLY short. NEVER turn a keyword into a question. NEVER add words like "benefits", "harms", "uses", "what is", "how to", "importance" unless they appear in the original.
- In "ur_en": write only the Urdu/grammatical words in Urdu script. Keep ALL English technical terms, proper nouns, and abbreviations in their ORIGINAL ENGLISH (Latin) spelling. DO NOT transliterate English words into Urdu script.
- In "roman": write the whole thing in Latin letters (Roman Urdu), keeping English technical terms as English.
- Keep abbreviations/proper nouns EXACTLY as in the query (UK, PrP, B12, IGF-1, DNA, etc.).
- Use ONLY Urdu-script characters and English (Latin) letters. NEVER use Chinese, Japanese, Korean, Cyrillic, or Hindi/Devanagari characters. This is mandatory.
- If a query is a single English word with nothing to translate, return it UNCHANGED in both fields.
- Keep the meaning identical.

EXAMPLES:
English: leeks
ur_en: leeks
roman: leeks

English: Cancer Risk From CT Scan Radiation
ur_en: CT Scan radiation سے cancer کا خطرہ
roman: CT Scan radiation se cancer ka khatra

English: What causes vitamin B12 deficiency?
ur_en: vitamin B12 deficiency کس وجہ سے ہوتی ہے؟
roman: vitamin B12 deficiency kis wajah se hoti hai?

Return ONLY a valid JSON array, same order:
[{"n": 1, "ur_en": "...", "roman": "..."}, {"n": 2, "ur_en": "...", "roman": "..."}]

Queries:
"""

def transform_batch(items, max_retries=6):
    numbered = "\n".join(f"{i+1}. {txt}" for i, (qid, txt) in enumerate(items))
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": BATCH_PROMPT + numbered}],
                temperature=0.2,
            )
            raw = resp.choices[0].message.content.strip().replace("```json","").replace("```","").strip()
            s, e = raw.find("["), raw.rfind("]")
            arr = json.loads(raw[s:e+1])
            out = {items[o["n"]-1][0]: {"ur_en": o["ur_en"], "roman": o["roman"]} for o in arr}
            if any(has_foreign(v["ur_en"]) or has_foreign(v["roman"]) for v in out.values()):
                print(f"    foreign chars detected, retry {attempt+1}...")
                time.sleep(3); continue
            return out
        except Exception as ex:
            print(f"    issue ({type(ex).__name__}), retry {attempt+1}...")
            time.sleep(4)
    raise RuntimeError("batch failed after retries")

def regenerate(ds_name, only_ids=None, batch_size=10, sleep_between=2):
    corpus, queries, qrels = GenericDataLoader(data_folder=f"data/{ds_name}").load(split="test")
    path = f"queries/{ds_name}_cs.jsonl"
    existing = {r["id"]: r for r in load_jsonl(path)} if os.path.exists(path) else {}
    target = list(only_ids) if only_ids is not None else list(queries.keys())
    items = [(qid, queries[qid]) for qid in target]
    nb = (len(items)+batch_size-1)//batch_size
    new = {}
    for b in range(0, len(items), batch_size):
        batch = items[b:b+batch_size]
        try:
            out = transform_batch(batch)
            new.update(out)
            print(f"  {ds_name} batch {b//batch_size+1}/{nb} OK ({len(out)})")
        except Exception as e:
            print(f"  {ds_name} batch {b//batch_size+1}/{nb} FAILED: {e}")
        time.sleep(sleep_between)
    for qid, v in new.items():
        existing[qid] = {"id": qid, "EN": queries[qid], "ur_en": v["ur_en"], "roman": v["roman"]}
    with open(path, "w", encoding="utf-8") as f:
        for qid in queries.keys():
            if qid in existing:
                f.write(json.dumps(existing[qid], ensure_ascii=False) + "\n")
    print(f"{ds_name}: regenerated {len(new)}, file total {len(existing)} -> {path}")

In [3]:
# 1) NFCorpus: full regenerate (removes keyword->question expansion)
print("=== Regenerating NFCorpus (no-expansion prompt) ===")
regenerate("nfcorpus")

# 2) SciFact: fix ONLY the foreign-char rows, keep the rest untouched
sci = load_jsonl("queries/scifact_cs.jsonl")
sci_bad = [r["id"] for r in sci if has_foreign(r["ur_en"]) or has_foreign(r["roman"])]
print(f"\n=== Fixing {len(sci_bad)} SciFact rows: {sci_bad} ===")
if sci_bad:
    regenerate("scifact", only_ids=sci_bad)

# 3) Final validation
print("\n=== Final scan ===")
for ds in ["scifact", "nfcorpus"]:
    data = load_jsonl(f"queries/{ds}_cs.jsonl")
    flagged = [r["id"] for r in data if has_foreign(r["ur_en"]) or has_foreign(r["roman"])]
    print(f"  {ds}: {len(data)} rows, foreign-char remaining: {len(flagged)} {flagged}")

=== Regenerating NFCorpus (no-expansion prompt) ===


100%|██████████| 3633/3633 [00:00<00:00, 144250.55it/s]


  nfcorpus batch 1/33 OK (10)
  nfcorpus batch 2/33 OK (10)
  nfcorpus batch 3/33 OK (10)
  nfcorpus batch 4/33 OK (10)
  nfcorpus batch 5/33 OK (10)
  nfcorpus batch 6/33 OK (10)
  nfcorpus batch 7/33 OK (10)
  nfcorpus batch 8/33 OK (10)
  nfcorpus batch 9/33 OK (10)
  nfcorpus batch 10/33 OK (10)
  nfcorpus batch 11/33 OK (10)
  nfcorpus batch 12/33 OK (10)
  nfcorpus batch 13/33 OK (10)
  nfcorpus batch 14/33 OK (10)
  nfcorpus batch 15/33 OK (10)
  nfcorpus batch 16/33 OK (10)
  nfcorpus batch 17/33 OK (10)
  nfcorpus batch 18/33 OK (10)
  nfcorpus batch 19/33 OK (10)
  nfcorpus batch 20/33 OK (10)
  nfcorpus batch 21/33 OK (10)
  nfcorpus batch 22/33 OK (10)
    foreign chars detected, retry 1...
    foreign chars detected, retry 2...
    foreign chars detected, retry 3...
    foreign chars detected, retry 4...
    foreign chars detected, retry 5...
    foreign chars detected, retry 6...
  nfcorpus batch 23/33 FAILED: batch failed after retries
    foreign chars detected, retry 1

100%|██████████| 5183/5183 [00:00<00:00, 132162.89it/s]


    foreign chars detected, retry 1...
    foreign chars detected, retry 2...
    foreign chars detected, retry 3...
    foreign chars detected, retry 4...
    foreign chars detected, retry 5...
    foreign chars detected, retry 6...
  scifact batch 1/3 FAILED: batch failed after retries
    foreign chars detected, retry 1...
    foreign chars detected, retry 2...
    foreign chars detected, retry 3...
    foreign chars detected, retry 4...
    foreign chars detected, retry 5...
    foreign chars detected, retry 6...
  scifact batch 2/3 FAILED: batch failed after retries
    foreign chars detected, retry 1...
    foreign chars detected, retry 2...
    foreign chars detected, retry 3...
    foreign chars detected, retry 4...
    foreign chars detected, retry 5...
    foreign chars detected, retry 6...
  scifact batch 3/3 FAILED: batch failed after retries
scifact: regenerated 0, file total 300 -> queries/scifact_cs.jsonl

=== Final scan ===
  scifact: 300 rows, foreign-char remaining: 2

In [4]:
def show_bad(ds):
    data = load_jsonl(f"queries/{ds}_cs.jsonl")
    bad = [r for r in data if has_foreign(r["ur_en"]) or has_foreign(r["roman"])]
    print(f"\n===== {ds}: {len(bad)} bad =====")
    for r in bad:
        print(f'id={r["id"]}')
        print('  EN :', r["EN"])
        print('  UR :', r["ur_en"])
        print('  ROM:', r["roman"])
    return bad
sci_bad = show_bad("scifact")
nf_bad  = show_bad("nfcorpus")


===== scifact: 24 bad =====
id=100
  EN : All hematopoietic stem cells segregate their chromosomes randomly.
  UR : تمام hematopoietic stem cells اپنے chromosomes کو随ائی طور پر الگ کرتے ہیں۔
  ROM: Tamam hematopoietic stem cells apne chromosomes ko rasmi tor par alag karte hain.
id=238
  EN : Cells undergoing methionine restriction may activate miRNAs.
  UR : methionine restriction سے گزرنے والے cells miRNAs کو सकریت کر سکتے ہیں
  ROM: methionine restriction se guzrne wale cells miRNAs ko saktif kar sakte hain
id=261
  EN : Chronic aerobic exercise alters endothelial function, improving vasodilating mechanisms mediated by NO.
  UR : دائمی aerobic ورزش endothelial function کو بدلتی ہے، NO کے ذریعے मधیہ کرنے والے vasodilating mechanisms کو بہتر بناتی ہے
  ROM: daimi aerobic wazarat endothelial function ko badalti hai, NO ke zariye madhyam karne wale vasodilating mechanisms ko behtar banati hai
id=327
  EN : Deletion of αvβ8 does not result in a spontaneous inflammatory phenotype.
  UR :

In [5]:
FIXES = {
    "100":  {"ur_en":"تمام hematopoietic stem cells اپنے chromosomes کو بے ترتیب طور پر الگ کرتے ہیں۔","roman":"Tamam hematopoietic stem cells apne chromosomes ko be-tarteeb tor par alag karte hain."},
    "238":  {"ur_en":"methionine restriction سے گزرنے والے cells miRNAs کو activate کر سکتے ہیں","roman":"methionine restriction se guzrne wale cells miRNAs ko activate kar sakte hain"},
    "261":  {"ur_en":"دائمی aerobic ورزش endothelial function کو بدلتی ہے، NO کے ذریعے mediate کرنے والے vasodilating mechanisms کو بہتر بناتی ہے","roman":"daimi aerobic warzish endothelial function ko badalti hai, NO ke zariye mediate karne wale vasodilating mechanisms ko behtar banati hai"},
    "560":  {"ur_en":"Immune responses کے نتیجے میں inflammatory Th17 cells اور anti-inflammatory iTregs کا development ہوتا ہے۔","roman":"Immune responses ke nateeje mein inflammatory Th17 cells aur anti-inflammatory iTregs ka development hota hai."},
    "628":  {"ur_en":"human T-cell lymphotropic virus type 1 کی infection African origin کے افراد میں سب سے زیادہ ہوتی ہے","roman":"human T-cell lymphotropic virus type 1 ki infection African origin ke afrad mein sab se zyada hoti hai"},
    "692":  {"ur_en":"Leuko-increased خون red blood cell transfusion میں infection کے مسائل کو بڑھاتا ہے","roman":"Leuko-increased khoon red blood cell transfusion mein infection ke masail ko barhata hai"},
    "693":  {"ur_en":"Leuko-reduced خون red blood cell transfusion میں infection کے مسائل کو کم کرتا ہے","roman":"Leuko-reduced khoon red blood cell transfusion mein infection ke masail ko kam karta hai"},
    "759":  {"ur_en":"ریاضیاتی ماڈلز کا اندازہ ہے کہ Artemisinin-based combination therapy کا استعمال nongametocytocidal drugs کے مقابلے میں malaria کی transmission کو کم کرنے میں ڈرامائی اثر ڈالتا ہے۔","roman":"riyaziati models ka andaza hai ke Artemisinin-based combination therapy ka istemaal nongametocytocidal drugs ke muqable mein malaria ki transmission ko kam karne mein dramatic asar dalta hai."},
    "820":  {"ur_en":"N-terminal cleavage transcription start sites کی پہچان کرنے میں کامیابی کو بڑھاتا ہے۔","roman":"N-terminal cleavage transcription start sites ki pehchaan karne mein kaamyaabi ko barhata hai."},
    "821":  {"ur_en":"N-terminal cleavage transcription start sites کی پہچان کرنے میں کامیابی کو کم کرتا ہے۔","roman":"N-terminal cleavage transcription start sites ki pehchaan karne mein kaamyaabi ko kam karta hai."},
    "847":  {"ur_en":"tuberculosis کے لئے نئے drugs اکثر tuberculosis lesion کے necrotic portion میں high concentrations میں penetrate نہیں کرتے۔","roman":"tuberculosis ke liye naye drugs aksar tuberculosis lesion ke necrotic portion mein high concentrations mein penetrate nahi karte."},
    "1107": {"ur_en":"Subcutaneous fat depots میں cold exposure کے بعد extensive browning processes ہوتے ہیں","roman":"Subcutaneous fat depots mein cold exposure ke baad extensive browning processes hotay hain"},
    "1180": {"ur_en":"PRR MDA5 RNA virus infection کا ایک sensor ہے۔","roman":"PRR MDA5 RNA virus infection ka ek sensor hai."},
    "1185": {"ur_en":"US health care system $750 million تک بچا سکتا ہے اگر kidney transplants کا انتظار کرنے والے 7% patients optimized national kidney paired donation program میں حصہ لیں۔","roman":"US health care system $750 million tak bacha sakta hai agar kidney transplants ka intezaar karne wale 7% patients optimized national kidney paired donation program mein hissa lien."},
    "1278": {"ur_en":"cancer patients کا co-IR blockade سے علاج کرنے سے کوئی adverse autoimmune events نہیں ہوتے","roman":"cancer patients ka co-IR blockade se ilaj karne se koi adverse autoimmune events nahi hote"},
    "1292": {"ur_en":"HNF4A mutations اور diabetes risks کے درمیان کوئی association نہیں ہے","roman":"HNF4A mutations aur diabetes risks ke darmiyan koi association nahi hai"},
    "1303": {"ur_en":"Tirasemtiv کا fast-twitch muscle پر کوئی effect نہیں ہے","roman":"Tirasemtiv ka fast-twitch muscle par koi effect nahi hai"},
    "1352": {"ur_en":"mosGCTL-1 کی upregulation West Nile virus سے infection پر ہوتی ہے","roman":"mosGCTL-1 ki upregulation West Nile virus se infection par hoti hai"},
    "PLAIN-2730": {"ur_en":"Anti-Angiogenesis: tumor کی supply lines کو کاٹنا","roman":"Anti-Angiogenesis: tumor ki supply lines ko kaatna"},
    "PLAIN-2500": {"ur_en":"Saturated Fat Studies: Buttering Up the Public","roman":"Saturated Fat Studies: Buttering Up the Public"},
    "PLAIN-3352": {"ur_en":"دل کی nerve control کو بڑھانا","roman":"dil ki nerve control ko barhana"},
}

# scanner now allows Greek (α β γ) and ® — they come from real source terms
def has_foreign(s):
    for ch in str(s):
        o = ord(ch)
        if o < 128: continue
        if 0x0600 <= o <= 0x06FF: continue
        if 0x0750 <= o <= 0x077F: continue
        if 0xFB50 <= o <= 0xFDFF: continue
        if 0xFE70 <= o <= 0xFEFF: continue
        if 0x0370 <= o <= 0x03FF: continue        # Greek (scientific)
        if ch in '‘’“”—–…•&®\u200c\u200d\u060c\u061f': continue
        return True
    return False

for ds in ["scifact","nfcorpus"]:
    path = f"queries/{ds}_cs.jsonl"
    data = load_jsonl(path)
    n=0
    for r in data:
        if r["id"] in FIXES:
            r["ur_en"] = FIXES[r["id"]]["ur_en"]
            r["roman"] = FIXES[r["id"]]["roman"]
            n+=1
    with open(path,"w",encoding="utf-8") as f:
        for r in data:
            f.write(json.dumps(r,ensure_ascii=False)+"\n")
    bad=[r["id"] for r in data if has_foreign(r["ur_en"]) or has_foreign(r["roman"])]
    print(f"{ds}: applied {n} fixes, foreign_left={len(bad)} {bad}")

print("\n--- my 21 fixes, EYEBALL these ---")
for ds in ["scifact","nfcorpus"]:
    for r in load_jsonl(f"queries/{ds}_cs.jsonl"):
        if r["id"] in FIXES:
            print(f'{r["id"]}: {r["roman"]}')

scifact: applied 18 fixes, foreign_left=0 []
nfcorpus: applied 3 fixes, foreign_left=0 []

--- my 21 fixes, EYEBALL these ---
100: Tamam hematopoietic stem cells apne chromosomes ko be-tarteeb tor par alag karte hain.
238: methionine restriction se guzrne wale cells miRNAs ko activate kar sakte hain
261: daimi aerobic warzish endothelial function ko badalti hai, NO ke zariye mediate karne wale vasodilating mechanisms ko behtar banati hai
560: Immune responses ke nateeje mein inflammatory Th17 cells aur anti-inflammatory iTregs ka development hota hai.
628: human T-cell lymphotropic virus type 1 ki infection African origin ke afrad mein sab se zyada hoti hai
692: Leuko-increased khoon red blood cell transfusion mein infection ke masail ko barhata hai
693: Leuko-reduced khoon red blood cell transfusion mein infection ke masail ko kam karta hai
759: riyaziati models ka andaza hai ke Artemisinin-based combination therapy ka istemaal nongametocytocidal drugs ke muqable mein malaria ki trans

In [6]:
import random
random.seed(1)
nf = load_jsonl("queries/nfcorpus_cs.jsonl")
for r in random.sample(nf, 15):
    print("EN   :", r["EN"]); print("ROM:", r["roman"]); print("---")

EN   : BRCA genes
ROM: BRCA genes
---
EN   : Is Milk and Mucus a Myth?
ROM: Milk aur Mucus ka mahsool ek khayal hai?
---
EN   : Didn't another study show carnitine was good for the heart?
ROM: kya aik aur study mein yeh nahi dikhaya gaya ke carnitine dil ke liye acha hai?
---
EN   : junk food
ROM: junk food
---
EN   : benzene
ROM: benzene
---
EN   : Preventing the Common Cold with Probiotics?
ROM: kya Probiotics ke saath common cold ko rokha ja sakta hai
---
EN   : Caloric Restriction vs. Plant-Based Diets
ROM: Caloric Restriction vs Plant-Based Diets
---
EN   : Alkylphenol Endocrine Disruptors and Allergies
ROM: Alkylphenol Endocrine Disruptors aur Allergies
---
EN   : Splenda
ROM: Splenda
---
EN   : flax oil
ROM: flax oil
---
EN   : American Dental Association
ROM: American Dental Association
---
EN   : Artificial Food Colors and ADHD
ROM: Artificial Food Colors aur ADHD
---
EN   : How Should I Take Probiotics?
ROM: Main Probiotics kis tarah le sakta hoon
---
EN   : suppositories
ROM

In [7]:
import random
random.seed(1)
nf = load_jsonl("queries/nfcorpus_cs.jsonl")
for r in random.sample(nf, 15):
    print("EN   :", r["EN"])
    print("UR_EN:", r["ur_en"])
    print("ROM  :", r["roman"])
    print("---")

EN   : BRCA genes
UR_EN: BRCA genes
ROM  : BRCA genes
---
EN   : Is Milk and Mucus a Myth?
UR_EN: Milk اور Mucus کا مفہوم ایک خیال ہے؟
ROM  : Milk aur Mucus ka mahsool ek khayal hai?
---
EN   : Didn't another study show carnitine was good for the heart?
UR_EN: کیا ایک اور study میں یہ نہیں دکھایا گیا کہ carnitine دل کے لیے اچھا ہے؟
ROM  : kya aik aur study mein yeh nahi dikhaya gaya ke carnitine dil ke liye acha hai?
---
EN   : junk food
UR_EN: junk food
ROM  : junk food
---
EN   : benzene
UR_EN: benzene
ROM  : benzene
---
EN   : Preventing the Common Cold with Probiotics?
UR_EN: کيا Probiotics کے ساتھ کامن کولڈ کو روکا جاسکتا ہے؟
ROM  : kya Probiotics ke saath common cold ko rokha ja sakta hai
---
EN   : Caloric Restriction vs. Plant-Based Diets
UR_EN: Caloric Restriction vs. Plant-Based Diets
ROM  : Caloric Restriction vs Plant-Based Diets
---
EN   : Alkylphenol Endocrine Disruptors and Allergies
UR_EN: Alkylphenol Endocrine Disruptors اور Allergies
ROM  : Alkylphenol Endocrine Disru

In [ ]:
import random
random.seed(1)
nf = load_jsonl("queries/nfcorpus_cs.jsonl")
for r in random.sample(nf, 15):
    print("EN   :", r["EN"]); print("ROMAN:", r["roman"]); print("---")